In [ ]:
# Importing libraries

import pandas as pd
import numpy as np
import requests # Allows API calls to ensembl
import time # Provides the sleep function for pausing between calls

In [ ]:
# Reading in filtered cis-pQTLs from original protein extractiong
snps = pd.read_csv("genetic_coloc_snps.csv")

In [6]:
# API function
def get_rsids_batch(chrom, pos_vector): # Takes a chromosome number and a list of positions and returns their rsIDs
    url = (f"https://rest.ensembl.org/overlap/region/human/" # Uses the GRCh38 endpoint (most recent and UKB-PPP has these locations
           f"{chrom}:{min(pos_vector)}-{max(pos_vector)}?feature=variation") # Creates the API URL for the position of the SNP of interest
    
    response = requests.get(url, headers={"Accept": "application/json"}) # Sends the API request to ensembl
    
    try:
        result = response.json()# changes json output into python dictionary
    except Exception: # Added this because there was a window where there was no output
        return None

    if not result:
        return None # skips the rest of the function if no rsIDs found (saves time)
    
    queries = set(f"{chrom}:{pos}" for pos in pos_vector) # Creates a set of chromosome:position strings for the rsIDs to be matched by
    rows = []
    for x in result: # Loops through every variant returned by the API
        if x.get("id", "").startswith("rs"): # Checking there is an rsID
            query = f"{chrom}:{x['start']}"
            if query in queries:
                rows.append({"query": query, "rsid": x["id"]}) # Checking if rsID is one of the ones needed
    
    return pd.DataFrame(rows) if rows else None #Returning results in a dataframe or none if there aren't any

In [7]:
# Creates a smaller column for just rsID lookup
snps_pos = snps[["CHROM", "GENPOS", "ID"]].copy()

# Creating a chromosome: position key for each SNP 
snps_pos["query"] = snps_pos["CHROM"].astype(str) + ":" + snps_pos["GENPOS"].astype(str)

In [ ]:
# Loop through chromosomes and windows

# Resume from checkpoint
checkpoint = pd.read_csv("rsids_checkpoint_genetic_coloc.csv")
all_rsids = []

# Find which chromosomes are already done
done_chroms = set(checkpoint["query"].str.split(":").str[0].astype(int).unique())
print(f"Already completed chromosomes: {sorted(done_chroms)}")

# Only run remaining chromosomes
remaining_chroms = [c for c in sorted(snps_pos["CHROM"].unique()) if c not in done_chroms]
print(f"Remaining chromosomes: {remaining_chroms}")

for chrom in remaining_chroms:
    print(f"Processing chromosome {chrom}") # Progress marker
    
    chr_snps = snps_pos[snps_pos["CHROM"] == chrom].copy() # Filters to just the SNPs on the current chromosome
    chr_snps["window"] = (chr_snps["GENPOS"] // 5000000).astype(int)  #  Splits the chromosome into 5Mb windows
    
    window_list = chr_snps["window"].unique() # Gets a list of unique windows
    total_windows = len(window_list) # Counts number of windows
    
    for i, w in enumerate(window_list): # Loops over each window, giving the value and the index
        print(f"  Chromosome {chrom} - window {i+1} of {total_windows}") # Progress marker
        batch = chr_snps[chr_snps["window"] == w] # Filters for SNPs just in that window
        time.sleep(0.1) # Gives the API time to sleep
        
        result = get_rsids_batch(chrom, batch["GENPOS"].tolist()) # Running function for extract proteins
        if result is not None:
            all_rsids.append(result) # Appending to list
            
    # Save after each chromosome completes
    checkpoint = pd.concat(all_rsids, ignore_index=True).drop_duplicates(subset="query")
    checkpoint.to_csv(f"rsids_checkpoint_genetic_coloc.csv", index=False)
    print(f"  Checkpoint saved after chromosome {chrom}")

Remaining chromosomes: [4, 5, 6, 7, 10, 11, 14, 16, 17, 19, 20, 21, 22]
Processing chromosome 4
  Chromosome 4 - window 1 of 1
  Checkpoint saved after chromosome 4
Processing chromosome 5
  Chromosome 5 - window 1 of 2
  Chromosome 5 - window 2 of 2
  Checkpoint saved after chromosome 5
Processing chromosome 6
  Chromosome 6 - window 1 of 2
  Chromosome 6 - window 2 of 2
  Checkpoint saved after chromosome 6
Processing chromosome 7
  Chromosome 7 - window 1 of 2
  Chromosome 7 - window 2 of 2
  Checkpoint saved after chromosome 7
Processing chromosome 10
  Chromosome 10 - window 1 of 1
  Checkpoint saved after chromosome 10
Processing chromosome 11
  Chromosome 11 - window 1 of 2
  Chromosome 11 - window 2 of 2
  Checkpoint saved after chromosome 11
Processing chromosome 14
  Chromosome 14 - window 1 of 2
  Chromosome 14 - window 2 of 2
  Checkpoint saved after chromosome 14
Processing chromosome 16
  Chromosome 16 - window 1 of 2
  Chromosome 16 - window 2 of 2
  Checkpoint saved aft

In [24]:
# Combine all results
rsid_dt = pd.concat(all_rsids, ignore_index=True).drop_duplicates(subset="query")
print(f"Found rsIDs for {len(rsid_dt)} positions")

Found rsIDs for 205591 positions


In [42]:
# Creating column that will match extracted rsIDS
snps_cis["query"] = snps_cis["CHROM"].astype(str) + ":" + snps_cis["GENPOS"].astype(str)

# Merge rsIDs back onto SNP data
snps_cis = snps_cis.merge(rsid_dt, on="query", how="inner")

Columns after merge: ['CHROM', 'GENPOS', 'ID', 'ALLELE0', 'ALLELE1', 'A1FREQ', 'INFO', 'N', 'TEST', 'BETA', 'SE', 'CHISQ', 'LOG10P', 'EXTRA', 'close_to_pqtl', 'source_protein', 'query', 'rsid']


In [43]:
print(f"SNPs with rsIDs: {len(snps_cis)}")
print(f"Proteins remaining: {snps_cis['source_protein'].nunique()}")

SNPs with rsIDs: 366318
Proteins remaining: 24


In [44]:
# number of snps by protein

snps_cis.groupby('source_protein').size()

source_protein
ACE         23103
ACHE        32154
APOC1       34914
APOE        34569
APP         22712
ARHGAP45    26864
BLNK        21788
CBLIF       10388
CD2AP       10967
CTF1         6205
DHRS4L2     10876
DMP1        10601
GNAS        11697
GRN          9076
LILRB2      15283
LRRC37A2     9565
PILRA       10245
PILRB        8377
PRSS8        5964
SERPINB1    12552
SERPINF2    14192
ST13         8340
STX4         6199
TXNDC15      9687
dtype: int64

In [ ]:
# Save output
snps_cis.to_csv("snps_genetic_coloc_rsids.csv", index=False)
print("Saved to snps_genetic_coloc_rsids.csv")

Saved to snps_genetic_coloc_rsids.csv
